# EU tourist accommodation: nights spent by month, region and guest origin

This notebook cleans and transforms Eurostat's official accommodation statistics, `tour_occ_nin2m`: nights in hotels, holiday and short-stay accommodation and campsites (NACE 55.1–55.3). 

#### Data sources (Eurostat)

- `data/raw/tour_occ_nin2m__custom_18882606_spreadsheet.xlsx`: nights spent at tourist accommodation establishments (NACE 55.1–55.3) by month, residence of the guest (domestic, foreign, total) and NUTS 2 region, **2023–2024** (extract of 7 November 2026). Each data sheet has a *Flags* sheet: `e` estimated, `u` low reliability, `C` confidential.
- `data/raw/NUTS2021-NUTS2024.xlsx`: NUTS codes, the NUTS 2021 → 2024 changes, NUTS 3 mountain and island typologies.
- `data/raw/EU-27-LAU-2024-NUTS-2024.xlsx`: municipalities with population, area, degree of urbanisation (DEGURBA) and coastal flag.
- `data/raw/estat_demo_r_pjanaggr3_filtered_en.csv` (population on 1 January by NUTS region, 2023–2024) and `data/raw/estat_reg_area3_en.csv` (area by NUTS region): Eurostat SDMX-CSV downloads, used for population and area (any file name containing the dataset code is found). Without them, Stage 1 falls back to sums over the municipality file, which lacks Iceland, Albania and Serbia.

#### Output: `data/transformed/tour_occ_nin2m_long.csv`

| Column | Meaning |
|---|---|
| Region | Name as published by Eurostat (NL35/NL36 named as in NUTS 2024) |
| NUTSCode | NUTS 2024 code (country code for level 0, `EL` = Greece) |
| NUTSLevel | 0 = country, 1 = NUTS 1, 2 = NUTS 2 |
| ParentRegion | NUTS 2 → its NUTS 1 region; NUTS 1 → its country; empty for countries |
| Country | Country name |
| Year | 2023, 2024 |
| Month | January … December, or `Total` (Eurostat's annual total) |
| ResidenceCountry | `Domestic`, `Foreign` or `Total` |
| Value | Nights spent; empty = not available (`:`) |
| Flag | `e` estimated, `u` low reliability, `C` confidential, empty = none |
| Days | Days in the month (28/29 … 31; 365/366 on `Total` rows) |
| NightsPerDay | Value ÷ Days |
| BoundaryNote | Set where a region is published under its NUTS 2021 code (see step 3) |
| Population | Residents on 1 January of the year (step 5b) |
| AreaKm2 | Total area, km² (step 5b) |
| PopAreaSource | Where population and area come from: `Eurostat demo_r_pjanaggr3 / reg_area3` or `LAU 2024 (municipality sums)` |
| PrcCities … PrcIsland | Geography of the region, % of residents (step 5) |

Steps: 1 catalogue · 2 parse values and flags · 3 scope, codes and hierarchy · 4 calendar days · 5 geography · 5b population and area · 6 consistency checks · 7 save · 8 tests against the source file.

In [1]:
import calendar
import time
import warnings

import numpy as np
import openpyxl
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

SRC = "data/raw/tour_occ_nin2m__custom_18882606_spreadsheet.xlsx"
NF = "data/raw/NUTS2021-NUTS2024.xlsx"
LF = "data/raw/EU-27-LAU-2024-NUTS-2024.xlsx"
OUT = "data/transformed/tour_occ_nin2m_long.csv"

sheets = pd.read_excel(SRC, sheet_name=None, header=None)
print(len(sheets), "sheets:", list(sheets))

14 sheets: ['Summary', 'Structure', 'Sheet 1', 'Flags 1', 'Sheet 2', 'Flags 2', 'Sheet 3', 'Flags 3', 'Sheet 4', 'Flags 4', 'Sheet 5', 'Flags 5', 'Sheet 6', 'Flags 6']


## 1. Sheet catalogue

The *Summary* sheet lists every data sheet and its flag sheet with the guest group and the year, so nothing is hard-coded.

In [2]:
summary = sheets["Summary"]
hdr = summary.index[summary[1].eq("Contents")][0]
catalog = summary.iloc[hdr + 1:, 1:7].dropna(subset=[1])
catalog.columns = ["Sheet", "Frequency", "Residence", "NACE", "Unit", "Year"]
catalog["Year"] = catalog["Year"].astype(int)
RESIDENCE = {"Domestic country [DOM]": "Domestic", "Foreign country [FOR]": "Foreign", "Total [TOTAL]": "Total"}
catalog["ResidenceCountry"] = catalog["Residence"].map(RESIDENCE)
assert catalog["ResidenceCountry"].notna().all()
data_sheets = catalog[catalog["Sheet"].str.startswith("Sheet")].copy()
data_sheets["FlagSheet"] = data_sheets["Sheet"].str.replace("Sheet", "Flags")
assert data_sheets["FlagSheet"].isin(catalog["Sheet"]).all(), "every data sheet needs its flag sheet"
data_sheets[["Sheet", "FlagSheet", "ResidenceCountry", "Year"]].reset_index(drop=True)

,Sheet,FlagSheet,ResidenceCountry,Year
0,Sheet 1,Flags 1,Domestic,2023
1,Sheet 2,Flags 2,Domestic,2024
2,Sheet 3,Flags 3,Foreign,2023
3,Sheet 4,Flags 4,Foreign,2024
4,Sheet 5,Flags 5,Total,2023
5,Sheet 6,Flags 6,Total,2024


## 2. Parse values and flags

Every data sheet has the same layout: GEO codes in column A, labels in column B, then the annual total and January–December. The flag sheet has exactly the same grid, so each value is paired with the flag in the same cell. `:` (not available) becomes an empty value — **not** zero.

In [3]:
def grid(df):
    """Rows of one sheet between 'GEO (Codes)' and the blank line before the legend, with month columns named."""
    start = df.index[df[0].eq("GEO (Codes)")][0] + 1
    month_row = df.index[df[0].eq("MONTH (Labels)")][0]
    block = df.loc[start:].copy()
    if block[0].isna().any():                                            # stop at the first empty row (if any)
        block = block.loc[: block[0].isna().idxmax() - 1]
    cols = {c: df.at[month_row, c] for c in df.columns[2:] if isinstance(df.at[month_row, c], str)}
    out = block[list(cols)].rename(columns=cols)
    out.insert(0, "Label", block[1].astype(str).str.strip())
    out.insert(0, "GEO", block[0].astype(str).str.strip())
    return out.reset_index(drop=True)

MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August",
          "September", "October", "November", "December"]
frames = []
for _, meta in data_sheets.iterrows():
    v, f = grid(sheets[meta["Sheet"]]), grid(sheets[meta["FlagSheet"]])
    assert v["GEO"].equals(f["GEO"]), f"{meta['Sheet']}: value and flag grids differ"
    val = v.melt(id_vars=["GEO", "Label"], value_vars=["Total"] + MONTHS, var_name="Month", value_name="Raw")
    flg = f.melt(id_vars=["GEO"], value_vars=["Total"] + MONTHS, var_name="Month", value_name="Flag")
    long = val.merge(flg, on=["GEO", "Month"])
    long["Year"], long["ResidenceCountry"] = meta["Year"], meta["ResidenceCountry"]
    frames.append(long)
raw = pd.concat(frames, ignore_index=True)
raw["Value"] = pd.to_numeric(raw["Raw"], errors="coerce")             # ':' -> NaN
raw["Flag"] = raw["Flag"].where(raw["Flag"].isin(["e", "u", "C"]))
print(f"{len(raw):,} cells, {raw['GEO'].nunique()} geo codes")
print("not available (':'):", int(raw["Value"].isna().sum()), "| flags:", raw["Flag"].value_counts().to_dict())

31,668 cells, 406 geo codes
not available (':'): 639 | flags: {'e': 120, 'u': 106, 'C': 88}


## 3. Scope, codes and hierarchy

* **NUTS level** = code length (2 country, 3 NUTS 1, 4 NUTS 2); **Country** = first two characters; **ParentRegion** = the code one level up. No name matching is needed.
* **Kept:** all 34 countries in the extract — EU-27, EFTA (Iceland, Liechtenstein, Norway, Switzerland) and the candidate countries North Macedonia, Albania and Serbia. Albania's 2023 values are Eurostat estimates (flag `e`); they are kept and flagged.
* **Left out, with reason:** the EU-27 aggregate (an aggregate, and it has no monthly values) and Portugal's NUTS 2021 rows PT16, PT17, PT18 (entirely empty — the NUTS 2024 regions PT19, PT1A–PT1D carry the data).
* **The Netherlands:** Utrecht and Zuid-Holland are published only under their **NUTS 2021** codes (NL31, NL33) and only for 2023. They are recoded to NUTS 2024 (NL35, NL36) so they match the map and the geography; the NUTS 2024 boundary shift between them is small (the municipality Vijfheerenlanden). `BoundaryNote` marks them.

In [4]:
CC = {"BE": "Belgium", "BG": "Bulgaria", "CZ": "Czechia", "DK": "Denmark", "DE": "Germany", "EE": "Estonia",
      "IE": "Ireland", "EL": "Greece", "ES": "Spain", "FR": "France", "HR": "Croatia", "IT": "Italy", "CY": "Cyprus",
      "LV": "Latvia", "LT": "Lithuania", "LU": "Luxembourg", "HU": "Hungary", "MT": "Malta", "NL": "Netherlands",
      "AT": "Austria", "PL": "Poland", "PT": "Portugal", "RO": "Romania", "SI": "Slovenia", "SK": "Slovakia",
      "FI": "Finland", "SE": "Sweden", "IS": "Iceland", "LI": "Liechtenstein", "NO": "Norway", "CH": "Switzerland",
      "MK": "North Macedonia", "AL": "Albania", "RS": "Serbia"}
RECODE = {"NL31": "NL35", "NL33": "NL36"}
NAME_2024 = {"NL35": "Utrecht", "NL36": "Zuid-Holland"}

d = raw[raw["GEO"].str[:2].isin(CC) & ~raw["GEO"].isin(["PT16", "PT17", "PT18"])].copy()
print("left out:", sorted(set(raw["GEO"]) - set(d["GEO"])))
d["BoundaryNote"] = np.where(d["GEO"].isin(RECODE), "Published under NUTS 2021 code " + d["GEO"] + "; 2023 only", None)
d["NUTSCode"] = d["GEO"].replace(RECODE)
d["Region"] = d["NUTSCode"].map(NAME_2024).fillna(d["Label"])
d["NUTSLevel"] = d["NUTSCode"].str.len() - 2
d["Country"] = d["NUTSCode"].str[:2].map(CC)
assert d["NUTSLevel"].isin([0, 1, 2]).all()

name_of = d.drop_duplicates("NUTSCode").set_index("NUTSCode")["Region"]
d["ParentRegion"] = np.select([d["NUTSLevel"] == 2, d["NUTSLevel"] == 1],
                              [d["NUTSCode"].str[:3].map(name_of), d["Country"]], None)
assert d.loc[d["NUTSLevel"] == 2, "ParentRegion"].notna().all(), "every NUTS 2 region needs its NUTS 1 parent"

# A series published with no value at all in a year (NL35/NL36 and Iceland in 2024) is left out of that year.
KEY_SERIES = ["NUTSCode", "Year", "ResidenceCountry"]
published = d.groupby(KEY_SERIES)["Value"].transform(lambda s: s.notna().any())
print("series with no published value, left out:")
display(d[~published].groupby(["NUTSCode", "Region", "Year"]).size().rename("cells").reset_index())
d = d[published].copy()
print(d.drop_duplicates("NUTSCode")["NUTSLevel"].value_counts().sort_index().rename("regions").to_dict(),
      "| countries:", d["Country"].nunique())

left out: ['EU27_2020', 'PT16', 'PT17', 'PT18']
series with no published value, left out:


,NUTSCode,Region,Year,cells
0,IS,Iceland,2024,39
1,IS0,Ísland,2024,39
2,IS00,Ísland,2024,39
3,NL35,Utrecht,2024,39
4,NL36,Zuid-Holland,2024,39


{0: 34, 1: 100, 2: 268} | countries: 34


## 4. Calendar days

Monthly totals mix two things: how busy a month is, and how long it is. February has 28 days (29 in 2024), so it holds about 10% fewer nights than a 31-day month even at the same daily level. Seasonality measures should describe the daily level, so **NightsPerDay** = nights ÷ days in the month (on the `Total` row: ÷ 365 or 366). Stage 2 computes every index from NightsPerDay. This follows the "intrinsic seasonal factors" adjusted for month length (Coshall, Charlesworth & Page 2015).

In [5]:
MONTH_NO = {m: i + 1 for i, m in enumerate(MONTHS)}
def days(row):
    y = int(row["Year"])
    return (366 if calendar.isleap(y) else 365) if row["Month"] == "Total" else calendar.monthrange(y, MONTH_NO[row["Month"]])[1]
dm = d[["Year", "Month"]].drop_duplicates()
dm["Days"] = dm.apply(days, axis=1)
d = d.merge(dm, on=["Year", "Month"])
d["NightsPerDay"] = (d["Value"] / d["Days"]).round(2)
dm.pivot(index="Year", columns="Month", values="Days")[MONTHS + ["Total"]]

Month,January,February,March,April,May,June,July,August,September,October,November,December,Total
Year,,,,,,,,,,,,,
2023,31,28,31,30,31,30,31,31,30,31,30,31,365
2024,31,29,31,30,31,30,31,31,30,31,30,31,366


## 5. Geography of the regions

Shares of each region's **residents** (2024 population) living in cities, rural areas, coastal municipalities, inland mountain NUTS 3 regions (mountain *and* not coastal) and island NUTS 3 regions — unchanged from the platform version. These describe where people live, not where guests stay; Stage 3 treats them as rough proxies.

In [6]:
nfx = pd.ExcelFile(NF)
xl = pd.ExcelFile(LF)
lau_sheets = [s for s in xl.sheet_names if s in CC]
lau = pd.concat([pd.read_excel(xl, sheet_name=s, usecols=["NUTS3", "POPULATION", "DEGURBA", "COASTAL AREA"])
                 for s in lau_sheets], ignore_index=True)
lau["POPULATION"] = pd.to_numeric(lau["POPULATION"], errors="coerce")
n_all = len(lau)
lau = lau[lau["DEGURBA"].isin([1, 2, 3]) & lau["POPULATION"].notna()].copy()
lau["NUTS3"] = lau["NUTS3"].astype(str).str.strip()
print(f"{len(lau_sheets)} countries, {n_all:,} municipalities, {len(lau):,} used, {lau['POPULATION'].sum() / 1e6:.1f} m residents")

typ = pd.read_excel(nfx, sheet_name="NUTS-3 Typologies")
typ["NUTS3"] = typ["NUTS-3 Code"].astype(str).str.strip()
known = typ["Mountanious regions"].notna()
typ["inland_mountain"] = np.where(known, (typ["Mountanious regions"].astype(str).str.startswith("mountain")
                                          & (typ["Coastal regions"] == "non-coastal")).astype(float), np.nan)
typ["island"] = np.where(typ["Island regions"].notna(), (typ["Island regions"] == "Island").astype(float), np.nan)
pop3 = lau.groupby("NUTS3")["POPULATION"].sum().rename("pop").reset_index().merge(
    typ[["NUTS3", "inland_mountain", "island"]], on="NUTS3", how="left")

parts = []
for n_chars in (2, 3, 4):                                   # country, NUTS 1, NUTS 2
    l_ = lau.assign(code=lau["NUTS3"].str[:n_chars])
    p = l_.pivot_table(index="code", columns="DEGURBA", values="POPULATION", aggfunc="sum").reindex(columns=[1, 2, 3]).fillna(0)
    c_ = l_.assign(cp=l_["POPULATION"] * l_["COASTAL AREA"].fillna(0)).groupby("code")[["cp", "POPULATION"]].sum()
    out = pd.DataFrame({"PrcCities": 100 * p[1] / p.sum(axis=1), "PrcRural": 100 * p[3] / p.sum(axis=1),
                        "PrcCoastal": 100 * c_["cp"] / c_["POPULATION"]})
    for flag, col in (("inland_mountain", "PrcInlandMountain"), ("island", "PrcIsland")):
        q = pop3.dropna(subset=[flag]).assign(code=lambda x: x["NUTS3"].str[:n_chars])
        q = q.assign(f=q["pop"] * q[flag]).groupby("code")[["f", "pop"]].sum()
        out[col] = 100 * q["f"] / q["pop"]
    parts.append(out)
GEO_COLS = ["PrcCities", "PrcRural", "PrcCoastal", "PrcInlandMountain", "PrcIsland"]
geo_shares = pd.concat(parts).round(2)[GEO_COLS]
d = d.merge(geo_shares, left_on="NUTSCode", right_index=True, how="left")
regs = d.drop_duplicates("NUTSCode")
print("regions without any geography values:", regs.loc[regs["PrcCities"].isna(), "Region"].tolist())
print("regions without mountain/island values:", regs.loc[regs["PrcIsland"].isna(), "Region"].tolist())

31 countries, 97,677 municipalities, 97,488 used, 469.3 m residents
regions without any geography values: ['Iceland', 'Ísland', 'Ísland', 'Albania', 'Shqipëria', 'Veri', 'Qender', 'Jug', 'Serbia', 'Srbija - sever', 'Beogradski region', 'Region Vojvodine', 'Srbija - jug', 'Region Sumadije i Zapadne Srbije', 'Region Juzne i Istocne Srbije']
regions without mountain/island values: ['Iceland', 'Ísland', 'Ísland', 'Liechtenstein', 'Liechtenstein', 'Liechtenstein', 'North Macedonia', 'Severna Makedonija', 'Severna Makedonija', 'Albania', 'Shqipëria', 'Veri', 'Qender', 'Jug', 'Serbia', 'Srbija - sever', 'Beogradski region', 'Region Vojvodine', 'Srbija - jug', 'Region Sumadije i Zapadne Srbije', 'Region Juzne i Istocne Srbije']


## 5b. Population and area

Needed for the tourism **intensity** (nights per 1,000 inhabitants) and **density** (nights per km²) indicators.

* **Preferred:** Eurostat's own regional tables, downloaded to `data/raw/` as SDMX-CSV with codes (any file name containing the dataset code, e.g. `estat_demo_r_pjanaggr3_filtered_en.csv`, `estat_reg_area3_en.csv`): `demo_r_pjanaggr3` (population on 1 January, all ages, both sexes) and `reg_area3` (total area, km²). These cover all 34 countries and give the population of each year.
* Eurostat publishes no area for Serbia in `reg_area3`, and a population of 0 for Svalbard (`NO0B`, no permanent residents): these stay empty.
* **Fallback:** sums over the municipalities of the LAU 2024 file (one reference population, total area of the municipalities). It has no Iceland, Albania or Serbia, which then stay empty.

In [7]:
import glob
pop_files, area_files = sorted(glob.glob("data/raw/*demo_r_pjanaggr3*.csv")), sorted(glob.glob("data/raw/*reg_area3*.csv"))   # any name containing the dataset code, e.g. estat_reg_area3_en.csv

def read_sdmx(path, **filters):
    t = pd.read_csv(path)
    t.columns = [c.split(":")[0].strip() for c in t.columns]       # 'geo: Geopolitical entity' -> 'geo'
    for k, v in filters.items():
        if k in t.columns:
            t = t[t[k].astype(str).str.split(":").str[0].str.strip() == v]
    t["geo"] = t["geo"].astype(str).str.split(":").str[0].str.strip()
    t["TIME_PERIOD"] = pd.to_numeric(t["TIME_PERIOD"], errors="coerce")
    return t[["geo", "TIME_PERIOD", "OBS_VALUE"]].dropna()

if pop_files and area_files:
    pop = read_sdmx(pop_files[-1], sex="T", age="TOTAL", unit="NR")
    pop = pop[pop["TIME_PERIOD"].isin([2023, 2024])].rename(columns={"geo": "NUTSCode", "TIME_PERIOD": "Year", "OBS_VALUE": "Population"})
    area = read_sdmx(area_files[-1], landuse="TOTAL", unit="KM2")
    area = area.sort_values("TIME_PERIOD").groupby("geo")["OBS_VALUE"].last().rename("AreaKm2")
    d = d.merge(pop, on=["NUTSCode", "Year"], how="left")
    d["AreaKm2"] = d["NUTSCode"].map(area)
    d["Population"] = d["Population"].where(d["Population"] > 0)      # Svalbard (NO0B): 0 residents published -> no intensity
    d["PopAreaSource"] = "Eurostat demo_r_pjanaggr3 / reg_area3"
    print("population and area from Eurostat:", pop_files[-1], area_files[-1])
else:
    lau_pa = pd.concat([pd.read_excel(xl, sheet_name=s_, usecols=["NUTS3", "POPULATION", "TOTAL AREA (m2)"]) for s_ in lau_sheets], ignore_index=True)
    lau_pa = lau_pa.apply(lambda c: pd.to_numeric(c, errors="coerce") if c.name != "NUTS3" else c.astype(str).str.strip())
    parts_pa = [lau_pa.assign(code=lau_pa["NUTS3"].str[:n]).groupby("code")[["POPULATION", "TOTAL AREA (m2)"]].sum() for n in (2, 3, 4)]
    pa = pd.concat(parts_pa)
    d["Population"] = d["NUTSCode"].map(pa["POPULATION"]).where(lambda x: x > 0)
    d["AreaKm2"] = (d["NUTSCode"].map(pa["TOTAL AREA (m2)"]) / 1e6).round(1).where(lambda x: x > 0)
    d["PopAreaSource"] = np.where(d["Population"].notna(), "LAU 2024 (municipality sums)", None)
    print("Eurostat population/area files not found in data/raw - using municipality sums from the LAU 2024 file")
regs = d.drop_duplicates("NUTSCode")
print("regions without population:", sorted(regs.loc[regs["Population"].isna(), "NUTSCode"]))
print("regions without area:", sorted(regs.loc[regs["AreaKm2"].isna(), "NUTSCode"]))
display(regs[regs["NUTSLevel"] == 0][["Country", "Population", "AreaKm2", "PopAreaSource"]].head(8))

population and area from Eurostat: data/raw\estat_demo_r_pjanaggr3_filtered_en.csv data/raw\estat_reg_area3_en.csv
regions without population: ['NO0B']
regions without area: ['RS', 'RS1', 'RS11', 'RS12', 'RS2', 'RS21', 'RS22']


,Country,Population,AreaKm2,PopAreaSource
0,Belgium,11742796.0,30667.0,Eurostat demo_r_pjanaggr3 / reg_area3
15,Bulgaria,6447710.0,110996.0,Eurostat demo_r_pjanaggr3 / reg_area3
24,Czechia,10827529.0,78871.0,Eurostat demo_r_pjanaggr3 / reg_area3
34,Denmark,5932654.0,42925.0,Eurostat demo_r_pjanaggr3 / reg_area3
41,Germany,83118501.0,357569.0,Eurostat demo_r_pjanaggr3 / reg_area3
96,Estonia,1365884.0,45336.0,Eurostat demo_r_pjanaggr3 / reg_area3
99,Ireland,5271395.0,69947.0,Eurostat demo_r_pjanaggr3 / reg_area3
104,Greece,10401868.0,131694.0,Eurostat demo_r_pjanaggr3 / reg_area3


## 6. Consistency checks

Differences of up to 10 nights are rounding in the source and are ignored.

1. The published annual `Total` equals the sum of the 12 months (where all 12 are published).
2. Domestic + Foreign = Total in every cell.
3. A country is at least the sum of its regions. *Country > regions* is expected for France (overseas regions with confidential months) the Netherlands (Utrecht and Zuid-Holland missing in 2024) and a few Spanish months (nights not allocated to a region).
4. Where the flags are: confidential and low-reliability cells by region.

In [8]:
TOL = 10                                         # Eurostat rounds confidential-safe values; differences up to ~10 nights are rounding
mon = d[d["Month"] != "Total"]
ms = mon.groupby(KEY_SERIES)["Value"].agg(["sum", "count"])
tot = d[d["Month"] == "Total"].set_index(KEY_SERIES)["Value"]
c1 = ms[ms["count"] == 12].join(tot.rename("total"))
c1 = c1[(c1["total"] - c1["sum"]).abs() > TOL]
print(f"1. annual total != sum of months: {len(c1)} series")

KEY_CELL = ["NUTSCode", "Year", "Month"]
w = d.pivot_table(index=KEY_CELL, columns="ResidenceCountry", values="Value")
gap = (w["Total"] - w["Domestic"] - w["Foreign"]).abs()
print(f"2. Domestic + Foreign != Total: {int((gap > TOL).sum())} cells")

KEY_COUNTRY = ["Country", "Year", "Month", "ResidenceCountry"]
country = d[d["NUTSLevel"] == 0].set_index(KEY_COUNTRY)["Value"]
for lvl in (1, 2):
    reg = d[d["NUTSLevel"] == lvl].groupby(KEY_COUNTRY)["Value"].sum(min_count=1)
    j = pd.concat([country.rename("country"), reg.rename("regions")], axis=1).dropna()
    diff = j["country"] - j["regions"]
    print(f"3. NUTS {lvl}: {len(j):,} cells | equal within {TOL} nights {diff.abs().le(TOL).sum():,} "
          f"| country > regions {(diff > TOL).sum():,} | regions > country {(diff < -TOL).sum():,}")
    print("   country > regions, by country (cells):", j[diff > TOL].reset_index()["Country"].value_counts().to_dict())

print("4. flagged and missing cells by region:")
display(d[d["Flag"].notna() | d["Value"].isna()].assign(missing=lambda x: x["Value"].isna())
        .groupby(["NUTSCode", "Region", "Year"]).agg(flags=("Flag", lambda s: ",".join(sorted(s.dropna().unique()))),
                                                     missing_cells=("missing", "sum")).reset_index())

1. annual total != sum of months: 0 series
2. Domestic + Foreign != Total: 0 cells
3. NUTS 1: 2,613 cells | equal within 10 nights 2,562 | country > regions 51 | regions > country 0
   country > regions, by country (cells): {'France': 39, 'Spain': 12}
3. NUTS 2: 2,613 cells | equal within 10 nights 2,499 | country > regions 114 | regions > country 0
   country > regions, by country (cells): {'France': 63, 'Netherlands': 39, 'Spain': 12}
4. flagged and missing cells by region:


,NUTSCode,Region,Year,flags,missing_cells
0,AL,Albania,2023,e,0
1,AL0,Shqipëria,2023,e,0
2,AL01,Veri,2023,e,0
3,AL02,Qender,2023,e,0
4,AL03,Jug,2023,e,0
5,FRM,Corse,2023,u,0
6,FRM0,Corse,2023,u,0
7,FRY1,Guadeloupe,2023,"C,u",2
8,FRY1,Guadeloupe,2024,u,24
9,FRY2,Martinique,2023,u,0


## 7. Save

In [9]:
MONTH_ORDER = MONTHS + ["Total"]
COUNTRY_ORDER = list(CC.values())
d["Month"] = pd.Categorical(d["Month"], categories=MONTH_ORDER, ordered=True)
d["Country"] = pd.Categorical(d["Country"], categories=COUNTRY_ORDER, ordered=True)
d = d.sort_values(["Country", "NUTSCode", "Year", "ResidenceCountry", "Month"]).reset_index(drop=True)
KEY = ["NUTSCode", "Year", "Month", "ResidenceCountry"]
assert not d.duplicated(KEY).any()
FINAL_COLS = ["Region", "NUTSCode", "NUTSLevel", "ParentRegion", "Country", "Year", "Month", "ResidenceCountry",
              "Value", "Flag", "Days", "NightsPerDay", "BoundaryNote", "Population", "AreaKm2", "PopAreaSource"] + GEO_COLS
d = d[FINAL_COLS]
d.to_csv(OUT, index=False, encoding="utf-8-sig")
print(f"saved {OUT}: {len(d):,} rows x {d.shape[1]} columns")
print(d.drop_duplicates("NUTSCode")["NUTSLevel"].value_counts().sort_index().rename("regions").to_dict(),
      "| countries:", d["Country"].nunique())

saved data/transformed/tour_occ_nin2m_long.csv: 31,161 rows x 21 columns
{0: 34, 1: 100, 2: 268} | countries: 34


## 8. Test: random comparison with the source file

The saved CSV is read back and random cells are compared with an **independent** read of the workbook with `openpyxl` (cell by cell, by GEO code), including the flag of the same cell. Fixed seed, so a run can be repeated.

In [10]:
SEED = 2026
rng = np.random.default_rng(SEED)
final = pd.read_csv(OUT, encoding="utf-8-sig", keep_default_na=False, na_values=[""])
csv_cell = final.set_index(["NUTSCode", "Year", "Month", "ResidenceCountry"])[["Value", "Flag"]]

wb = openpyxl.load_workbook(SRC, data_only=True)
_cache = {}
def sheet_cells(name):
    if name not in _cache:
        rows = [list(r) for r in wb[name].iter_rows(values_only=True)]
        labs = [str(r[0]).strip() if r[0] is not None else None for r in rows]
        mrow = labs.index("MONTH (Labels)")
        months = {str(v).strip(): j for j, v in enumerate(rows[mrow]) if j > 1 and v not in (None, "")}
        geo_row = {labs[i]: i for i in range(labs.index("GEO (Codes)") + 1, len(labs)) if labs[i]}
        _cache[name] = (rows, months, geo_row)
    return _cache[name]

sheet_of = {(r["ResidenceCountry"], r["Year"]): (r["Sheet"], r["FlagSheet"]) for _, r in data_sheets.iterrows()}
INV = {v: k for k, v in RECODE.items()}
bad, samp = [], final.sample(600, random_state=SEED)
for _, r in samp.iterrows():
    vs, fs = sheet_of[(r["ResidenceCountry"], int(r["Year"]))]
    rows, months, geo_row = sheet_cells(vs)
    frows, fmonths, fgeo = sheet_cells(fs)
    code = INV.get(r["NUTSCode"], r["NUTSCode"])
    v = rows[geo_row[code]][months[r["Month"]]]
    fl = frows[fgeo[code]][fmonths[r["Month"]]]
    v = None if v in (None, ":") else float(v)
    fl = fl if fl in ("e", "u", "C") else None
    ok_v = (v is None and pd.isna(r["Value"])) or (v is not None and abs(v - r["Value"]) < 0.5)
    ok_f = (fl is None and pd.isna(r["Flag"])) or fl == r["Flag"]
    if not (ok_v and ok_f):
        bad.append({"cell": (r["NUTSCode"], r["Year"], r["Month"], r["ResidenceCountry"]), "csv": (r["Value"], r["Flag"]), "source": (v, fl)})
print(f"A. CSV vs workbook: {len(samp)} cells checked, {len(bad)} mismatches")
assert not bad, pd.DataFrame(bad).head()

checks = {
    "unique key": not final.duplicated(["NUTSCode", "Year", "Month", "ResidenceCountry"]).any(),
    "13 rows per series": (final.groupby(["NUTSCode", "Year", "ResidenceCountry"]).size() == 13).all(),
    "34 countries": final["Country"].nunique() == 34,
    "population and area positive where present": (final["Population"].dropna() > 0).all() and (final["AreaKm2"].dropna() > 0).all(),
    "no negative values": (final["Value"].dropna() >= 0).all(),
    "NightsPerDay = Value / Days": np.allclose((final["Value"] / final["Days"]).dropna(), final["NightsPerDay"].dropna(), atol=0.01),
    "flags only e/u/C": final["Flag"].dropna().isin(["e", "u", "C"]).all(),
    "e only in Albania 2023": set(final.loc[final["Flag"] == "e", "Country"]) <= {"Albania"},
    "geography shares within 0-100": final[GEO_COLS].stack().dropna().between(0, 100).all(),
}
display(pd.Series(checks, name="passed").to_frame())
assert all(checks.values())
print(f"All checks passed (seed {SEED}).")

A. CSV vs workbook: 600 cells checked, 0 mismatches


,passed
unique key,True
13 rows per series,True
34 countries,True
population and area positive where present,True
no negative values,True
NightsPerDay = Value / Days,True
flags only e/u/C,True
e only in Albania 2023,True
geography shares within 0-100,True


All checks passed (seed 2026).
